# 03 — Model Building
**Fraud Detection & Anomaly Scoring System**

Goals:
- Train Isolation Forest (unsupervised)
- Train XGBoost with Optuna hyperparameter search
- Train LightGBM with Optuna hyperparameter search
- Save all models to disk

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

from src.features import build_train_test
from src.models import (
    train_isolation_forest, iso_forest_scores,
    train_xgboost, train_lightgbm,
)

plt.rcParams.update({'figure.dpi': 120})

print("[1/4] Building features …")
X_train, X_val, X_test, y_train, y_val, y_test, feat_names = build_train_test()
print(f"  X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

## 1. Isolation Forest

In [ ]:
iso = train_isolation_forest(X_train)

# Score distribution
scores = iso_forest_scores(iso, X_test)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores[y_test==0], bins=80, alpha=0.5, color='#27ae60', label='Legitimate', density=True)
ax.hist(scores[y_test==1], bins=80, alpha=0.7, color='#e74c3c', label='Fraud', density=True)
ax.set_title('Isolation Forest Anomaly Scores', fontsize=13, fontweight='bold')
ax.set_xlabel('Anomaly Score (higher = more anomalous)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/model_isoforest_scores.png', dpi=120)
plt.show()

## 2. XGBoost

In [ ]:
# Set n_trials=40 for a proper run; use 10 here for a quick demo
xgb_model = train_xgboost(X_train, y_train, X_val, y_val, n_trials=20)

# Feature importance
import pandas as pd
fi = pd.Series(xgb_model.feature_importances_, index=feat_names).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
fi.plot.bar(ax=ax, color='#3498db', edgecolor='white')
ax.set_title('XGBoost — Top 15 Feature Importances', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('../reports/model_xgb_feature_importance.png', dpi=120)
plt.show()

## 3. LightGBM

In [ ]:
lgbm_model = train_lightgbm(X_train, y_train, X_val, y_val, n_trials=20)

fi_lgbm = pd.Series(lgbm_model.feature_importances_, index=feat_names).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 5))
fi_lgbm.plot.bar(ax=ax, color='#9b59b6', edgecolor='white')
ax.set_title('LightGBM — Top 15 Feature Importances', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('../reports/model_lgbm_feature_importance.png', dpi=120)
plt.show()

## 4. Summary

All three models saved to `models/` directory.

| Model | Type | Key Strength |
|-------|------|--------------|
| Isolation Forest | Unsupervised | No labels needed, fast |
| XGBoost | Supervised | High accuracy, interpretable |
| LightGBM | Supervised | Faster training, lower memory |

> **Next step:** Evaluation (`04_evaluation.ipynb`)